<a href="https://colab.research.google.com/github/shafiq73/2024/blob/main/New_Ydata_mini_report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
import plotly.express as px
from jinja2 import Template

def smart_profiling_report(df, filename="Smart_EDA_Report.html"):
    # Bare datasets ke liye sampling (Performance ke liye)
    if len(df) > 50000:
        plot_df = df.sample(50000, random_state=42)
    else:
        plot_df = df

    # ========== BASIC INFO ==========
    shape = df.shape
    dtypes_html = df.dtypes.to_frame("Data Type").to_html(classes='table table-striped')
    describe_html = df.describe(include='all').to_html(classes='table table-hover')

    # ========== MISSING VALUES ==========
    missing = (df.isnull().sum() / len(df) * 100).reset_index()
    missing.columns = ['Column', 'Missing %']
    fig_missing = px.bar(missing, x='Column', y='Missing %',
                         title="Missing Values %", color_discrete_sequence=['#ef553b'])
    missing_plot = fig_missing.to_html(full_html=False)

    # ========== CORRELATION ==========
    corr = df.corr(numeric_only=True)
    fig_corr = px.imshow(corr, text_auto=True, title="Correlation Heatmap", color_continuous_scale='RdBu_r')
    corr_plot = fig_corr.to_html(full_html=False)

    high_corr = []
    for i in corr.columns:
        for j in corr.columns:
            if i != j and abs(corr.loc[i, j]) > 0.8:
                high_corr.append(f"{i} & {j} : {corr.loc[i,j]:.2f}")

    # ========== HISTOGRAMS & OUTLIERS ==========
    histograms = ""
    outlier_alerts = []
    numeric_cols = df.select_dtypes(include=np.number).columns

    for col in numeric_cols:
        # Histogram
        fig = px.histogram(plot_df, x=col, title=f"Distribution of {col}", marginal="box")
        histograms += fig.to_html(full_html=False)

        # Outlier Detection (IQR Method)
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        outliers_count = len(df[(df[col] < (q1 - 1.5 * iqr)) | (df[col] > (q3 + 1.5 * iqr))])
        if outliers_count > 0:
            outlier_alerts.append(f"{col}: {outliers_count} outliers detected.")

    # ========== CATEGORICAL BARS ==========
    bars = ""
    for col in df.select_dtypes(include='object').columns:
        vc = df[col].value_counts().head(10).reset_index()
        vc.columns = [col, 'Count']
        fig = px.bar(vc, x=col, y='Count', title=f"Top 10 Categories: {col}", color_discrete_sequence=['#636efa'])
        bars += fig.to_html(full_html=False)

    # ========== ALERTS / WARNINGS ==========
    alerts = []
    const_cols = [col for col in df.columns if df[col].nunique() == 1]
    if const_cols: alerts.append(f"<b>Constant Columns:</b> {const_cols}")

    high_card = [col for col in df.columns if df[col].nunique() > 50 and df[col].dtype == 'object']
    if high_card: alerts.append(f"<b>High Cardinality:</b> {high_card}")

    miss_cols = missing[missing['Missing %'] > 0]['Column'].tolist()
    if miss_cols: alerts.append(f"<b>Missing Values:</b> {miss_cols}")

    if high_corr: alerts.append(f"<b>High Correlation:</b> {list(set(high_corr[:5]))}...") # Limit top 5

    if outlier_alerts: alerts.append(f"<b>Outliers:</b> {outlier_alerts[:3]}...")

    alerts_html = "".join([f"<div class='alert'>{a}</div>" for a in alerts]) if alerts else "No major warnings."

    # ========== HTML TEMPLATE WITH CSS ==========
    html_template = """
    <html>
    <head>
        <title>Smart EDA Report</title>
        <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.2.3/dist/css/bootstrap.min.css">
        <style>
            body { background-color: #f8f9fa; padding: 20px; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; }
            .container-custom { background: white; padding: 30px; border-radius: 15px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }
            h1 { color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; margin-bottom: 30px; }
            h2 { color: #2980b9; margin-top: 40px; border-left: 5px solid #3498db; padding-left: 10px; }
            .alert { background: #fff3cd; border-left: 5px solid #ffc107; margin-bottom: 10px; padding: 10px; border-radius: 5px; }
            .table { font-size: 0.9rem; margin-top: 15px; }
            .plot-container { margin-bottom: 50px; }
        </style>
    </head>
    <body>
        <div class="container container-custom">
            <h1>📊 Smart Auto EDA Report</h1>

            <div class="row">
                <div class="col-md-4">
                    <h3>Dataset Shape</h3>
                    <p class="lead">Rows: <b>{{shape[0]}}</b> | Cols: <b>{{shape[1]}}</b></p>
                </div>
                <div class="col-md-8">
                    <h3>⚠️ Alerts & Warnings</h3>
                    {{alerts}}
                </div>
            </div>

            <h2>📋 Data Types & Summary</h2>
            <div class="row">
                <div class="col-md-4">{{dtypes}}</div>
                <div class="col-md-8" style="overflow-x: auto;">{{describe}}</div>
            </div>

            # <h2>📉 Missing Values Analysis</h2>
            # <div class="plot-container">{{missing_plot}}</div>

            <h2>🔥 Correlation Heatmap</h2>
            <div class="plot-container">{{corr_plot}}</div>

            <h2>🔢 Numeric Distributions (with Boxplots)</h2>
            <div class="plot-container">{{histograms}}</div>

            <h2>🔠 Categorical Analysis</h2>
            <div class="plot-container">{{bars}}</div>
        </div>
    </body>
    </html>
    """

    template = Template(html_template)
    html = template.render(shape=shape, alerts=alerts_html, dtypes=dtypes_html,
                           describe=describe_html, missing_plot=missing_plot,
                           corr_plot=corr_plot, histograms=histograms, bars=bars)

    with open(filename, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"✅ Professional EDA report saved as: {filename}")

# --- Test Karne ke liye ---
if __name__ == "__main__":
    # Sample dataset (Titanic) use kar rahe hain test ke liye
    df_test = px.data.tips()
    smart_profiling_report(df_test)

✅ Professional EDA report saved as: Smart_EDA_Report.html


In [9]:
import pandas as pd
import numpy as np
import plotly.express as px
from jinja2 import Template
from google.colab import files  # Google Colab download function

def smart_profiling_report(df, filename="Smart_EDA_Report1.html"):
    """
    User logic + Advanced Styling + Colab Download
    """
    # ========== BASIC INFO ==========
    shape = df.shape
    dtypes_html = df.dtypes.to_frame("Data Type").to_html(classes='table table-hover')
    describe_html = df.describe(include='all').to_html(classes='table table-striped')

    # ========== MISSING VALUES ==========
    missing = (df.isnull().sum() / len(df) * 100).reset_index()
    missing.columns = ['Column', 'Missing %']
    fig_missing = px.bar(missing, x='Column', y='Missing %',
                         title="Missing Values Analysis", color_discrete_sequence=['#FF4B4B'])
    missing_plot = fig_missing.to_html(full_html=False)

    # ========== CORRELATION ==========
    corr = df.corr(numeric_only=True)
    fig_corr = px.imshow(corr, text_auto=True, title="Correlation Matrix", color_continuous_scale='RdBu_r')
    corr_plot = fig_corr.to_html(full_html=False)

    # ========== ALERTS / WARNINGS ==========
    alerts = []
    const_cols = [col for col in df.columns if df[col].nunique() == 1]
    if const_cols: alerts.append(f"<b>Constant Columns:</b> {const_cols}")

    miss_cols = missing[missing['Missing %'] > 0]['Column'].tolist()
    if miss_cols: alerts.append(f"<b>Missing Values:</b> {miss_cols}")

    # High Correlation detection
    high_corr = []
    for i in corr.columns:
        for j in corr.columns:
            if i != j and abs(corr.loc[i, j]) > 0.8:
                high_corr.append(f"{i} & {j}")
    if high_corr: alerts.append(f"<b>High Correlation Pairs:</b> {list(set(high_corr[:5]))}")

    alerts_html = "".join([f"<div class='alert alert-warning'>{a}</div>" for a in alerts]) if alerts else "No major warnings."

    # ========== DISTRIBUTIONS (NUMERIC) ==========
    histograms = ""
    for col in df.select_dtypes(include=np.number).columns:
        fig = px.histogram(df, x=col, title=f"Distribution: {col}", marginal="box")
        histograms += fig.to_html(full_html=False)

    # ========== CATEGORICAL BARS ==========
    bars = ""
    for col in df.select_dtypes(include='object').columns:
        vc = df[col].value_counts().head(10).reset_index()
        vc.columns = [col, 'Count']
        fig = px.bar(vc, x=col, y='Count', title=f"Top Categories: {col}")
        bars += fig.to_html(full_html=False)

    # ========== HTML TEMPLATE (The Design) ==========
    html_template = """
    <html>
    <head>
        <title>Smart EDA Report</title>
        <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.2.3/dist/css/bootstrap.min.css">
        <style>
            body { background-color: #f0f2f6; padding: 30px; font-family: sans-serif; }
            .report-card { background: white; padding: 25px; border-radius: 15px; box-shadow: 0 10px 25px rgba(0,0,0,0.05); margin-bottom: 30px; }
            h1 { color: #1f77b4; font-weight: bold; border-bottom: 3px solid #1f77b4; display: inline-block; padding-bottom: 5px; }
            .table-container { overflow-x: auto; }
            .alert-warning { border-left: 5px solid #ffa421; }
        </style>
    </head>
    <body>
        <div class="container">
            <div class="text-center mb-5">
                <h1>📊 Smart Auto EDA Report</h1>
                <p class="text-muted">Generated automatically for Colab Analysis</p>
            </div>

            <div class="row report-card">
                <div class="col-md-4">
                    <h3>Dataset Shape</h3>
                    <p class="display-6"><b>{{shape[0]}}</b> <small>Rows</small></p>
                    <p class="display-6"><b>{{shape[1]}}</b> <small>Cols</small></p>
                </div>
                <div class="col-md-8">
                    <h3>⚠️ Alerts & Warnings</h3>
                    {{alerts}}
                </div>
            </div>

            <div class="report-card">
                <h3>📋 Data Summary</h3>
                <div class="row">
                    <div class="col-md-4"><h5>Types</h5>{{dtypes}}</div>
                    <div class="col-md-8"><h5>Stats</h5><div class="table-container">{{describe}}</div></div>
                </div>
            </div>

            <div class="report-card">
                <h3>📉 Visual Analysis</h3>
                {{missing_plot}}
                {{corr_plot}}
                {{histograms}}
                {{bars}}
            </div>
        </div>
    </body>
    </html>
    """

    # Template Rendering
    template = Template(html_template)
    html = template.render(shape=shape, alerts=alerts_html, dtypes=dtypes_html,
                           describe=describe_html, missing_plot=missing_plot,
                           corr_plot=corr_plot, histograms=histograms, bars=bars)

    # 1. File local folder mein save hogi
    with open(filename, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"✅ Report '{filename}' ")

    # 2. auto-download trigger
    files.download(filename)

# --- EXECUTION ---
#  default dataset load
try:
    df = pd.read_csv('google_play_store_updated.csv')
    smart_profiling_report(df)
except FileNotFoundError:
    print("Dataset nahi mila, please path check karein.")

✅ Report 'Smart_EDA_Report1.html' taiyar hai.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>